In [1]:
import os
import sys

# ── windows environment setup ─────────────────────────────────────────
os.environ["JAVA_HOME"]             = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["HADOOP_HOME"]           = r"D:\hadoop"
os.environ["PATH"]                  += os.pathsep + r"D:\hadoop\bin"
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"]        = "127.0.0.1"

print("=" * 55)
print("STEP 1 — ENVIRONMENT SETUP")
print("=" * 55)
print(f"✅ JAVA_HOME    : {os.environ['JAVA_HOME']}")
print(f"✅ HADOOP_HOME  : {os.environ['HADOOP_HOME']}")
print(f"✅ PYTHON       : {sys.executable}")
print(f"✅ SPARK IP     : {os.environ['SPARK_LOCAL_IP']}")
print("=" * 55)
print("🚀 Environment ready!")

STEP 1 — ENVIRONMENT SETUP
✅ JAVA_HOME    : C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
✅ HADOOP_HOME  : D:\hadoop
✅ PYTHON       : C:\Users\easho\anaconda3\envs\pyspark_env\python.exe
✅ SPARK IP     : 127.0.0.1
🚀 Environment ready!


In [2]:
from pyspark.sql import SparkSession
import pyspark

# ── start spark session ───────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("Airline_Ingestion") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("=" * 55)
print("STEP 2 — SPARK SESSION")
print("=" * 55)
print(f"✅ Spark version   : {spark.version}")
print(f"✅ PySpark version : {pyspark.__version__}")
print(f"✅ Spark UI        : {spark.sparkContext.uiWebUrl}")
print(f"✅ Total CPU cores : {spark.sparkContext.defaultParallelism}")
print(f"✅ Driver memory   : 4 GB")
print(f"✅ App name        : {spark.sparkContext.appName}")
print("=" * 55)
print("🚀 Spark is running!")
print(f"   Open Spark UI → http://localhost:4040")

STEP 2 — SPARK SESSION
✅ Spark version   : 4.1.1
✅ PySpark version : 4.1.1
✅ Spark UI        : http://view-localhost:4040
✅ Total CPU cores : 12
✅ Driver memory   : 4 GB
✅ App name        : Airline_Ingestion
🚀 Spark is running!
   Open Spark UI → http://localhost:4040


In [3]:
import os

RAW_DIR       = r"D:\airline_pipeline\data\raw"
PROCESSED_DIR = r"D:\airline_pipeline\data\processed"
PARQUET_PATH  = PROCESSED_DIR + r"\flights_clean.parquet"

os.makedirs(PROCESSED_DIR, exist_ok=True)

# ── scan raw files ────────────────────────────────────────────────────
csv_files = sorted([
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR)
    if f.endswith(".csv")
])

total_size_gb = sum(os.path.getsize(f) for f in csv_files) / (1024**3)

# ── get years available ───────────────────────────────────────────────
years = sorted(set(
    f.split("present)_")[1].split("_")[0]
    for f in os.listdir(RAW_DIR)
    if f.endswith(".csv") and "present)" in f
))

print("=" * 55)
print("STEP 3 — RAW DATA FILES CHECK")
print("=" * 55)
print(f"✅ Total CSV files     : {len(csv_files)}")
print(f"✅ Total size on disk  : {total_size_gb:.2f} GB")
print(f"✅ Location            : {RAW_DIR}")
print(f"✅ Years available     :", end=" ")
for y in years:
    count = sum(1 for f in os.listdir(RAW_DIR)
                if f.endswith(".csv") and f"present)_{y}_" in f)
    print(f"{y}({count}mo)", end=" ")
print()
print("=" * 55)
print(f"\n📄 First 3 files:")
for f in csv_files[:3]:
    size = os.path.getsize(f) / (1024*1024)
    print(f"   {os.path.basename(f)}  ({size:.1f} MB)")
print(f"\n📄 Last 3 files:")
for f in csv_files[-3:]:
    size = os.path.getsize(f) / (1024*1024)
    print(f"   {os.path.basename(f)}  ({size:.1f} MB)")
print("=" * 55)
print("✅ Raw data check complete!")

STEP 3 — RAW DATA FILES CHECK
✅ Total CSV files     : 47
✅ Total size on disk  : 11.40 GB
✅ Location            : D:\airline_pipeline\data\raw
✅ Years available     : 2022(12mo) 2023(12mo) 2024(12mo) 2025(11mo) 

📄 First 3 files:
   On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_1.csv  (229.9 MB)
   On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_10.csv  (246.6 MB)
   On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_11.csv  (235.7 MB)

📄 Last 3 files:
   On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_7.csv  (272.9 MB)
   On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_8.csv  (260.1 MB)
   On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_9.csv  (242.3 MB)
✅ Raw data check complete!


In [4]:
import time
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

print("=" * 55)
print("STEP 4 — LOADING CSV FILES INTO SPARK")
print("=" * 55)
print(f"⏳ Loading {len(csv_files)} CSV files...")
print("   (using samplingRatio=0.01 for fast schema detection)\n")

start = time.time()

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("samplingRatio", "0.01") \
    .csv(csv_files)

# ── select only the columns we need ──────────────────────────────────
cols_we_need = [
    "Year", "Quarter", "Month", "DayofMonth", "DayOfWeek",
    "FlightDate", "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin", "OriginCityName", "OriginState",
    "Dest", "DestCityName", "DestState",
    "CRSDepTime", "DepTime", "DepDelay", "DepDelayMinutes",
    "CRSArrTime", "ArrTime", "ArrDelay", "ArrDelayMinutes",
    "Cancelled", "CancellationCode", "Diverted", "Distance",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
]
existing = [c for c in cols_we_need if c in df_raw.columns]
df_raw   = df_raw.select(existing)

duration = round(time.time() - start, 2)

print(f"✅ Total columns in CSV    : 110")
print(f"✅ Columns selected        : {len(existing)}")
print(f"✅ CSV files loaded        : {len(csv_files)}")
print(f"✅ Raw data size on disk   : {total_size_gb:.2f} GB")
print(f"⏱️  Load time               : {duration} seconds")
print("=" * 55)
print("\n📋 SCHEMA:")
df_raw.printSchema()

STEP 4 — LOADING CSV FILES INTO SPARK
⏳ Loading 47 CSV files...
   (using samplingRatio=0.01 for fast schema detection)

✅ Total columns in CSV    : 110
✅ Columns selected        : 31
✅ CSV files loaded        : 47
✅ Raw data size on disk   : 11.40 GB
⏱️  Load time               : 50.07 seconds

📋 SCHEMA:
root
 |-- Year: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepTime: integer (nullable = true)
 |--

In [5]:
print("=" * 55)
print("STEP 5 — DATA VALIDATION (NULL CHECK)")
print("=" * 55)

key_cols = [
    "Year", "Month", "Origin", "Dest",
    "Reporting_Airline", "DepDelay", "ArrDelay",
    "Distance", "CarrierDelay", "WeatherDelay",
    "NASDelay", "Cancelled"
]
existing_cols = [c for c in key_cols if c in df_raw.columns]

null_exprs  = [F.sum(F.col(c).isNull().cast("int")).alias(c)
               for c in existing_cols]
null_counts = df_raw.select(null_exprs).collect()[0]

print(f"{'Column':<28} {'Nulls':>10}  Status")
print("-" * 55)
for col in existing_cols:
    count  = null_counts[col]
    status = "✅ OK" if count == 0 else "⚠️  has nulls"
    print(f"  {col:<26} {str(count):>10}  {status}")

print("=" * 55)
print("✅ Validation complete!")

STEP 5 — DATA VALIDATION (NULL CHECK)
Column                            Nulls  Status
-------------------------------------------------------
  Year                                0  ✅ OK
  Month                               0  ✅ OK
  Origin                              0  ✅ OK
  Dest                                0  ✅ OK
  Reporting_Airline                   0  ✅ OK
  DepDelay                       444792  ⚠️  has nulls
  ArrDelay                       527178  ⚠️  has nulls
  Distance                            0  ✅ OK
  CarrierDelay                 21480415  ⚠️  has nulls
  WeatherDelay                 21480415  ⚠️  has nulls
  NASDelay                     21480415  ⚠️  has nulls
  Cancelled                           0  ✅ OK
✅ Validation complete!


In [6]:
print("=" * 55)
print("STEP 6 — DATA CLEANING")
print("=" * 55)

df_clean = df_raw \
    .dropna(subset=["Year", "Month", "Origin",
                    "Dest", "Reporting_Airline"]) \
    .fillna({
        "CarrierDelay": 0.0, "WeatherDelay": 0.0,
        "NASDelay": 0.0, "SecurityDelay": 0.0,
        "LateAircraftDelay": 0.0, "DepDelay": 0.0,
        "ArrDelay": 0.0, "DepDelayMinutes": 0.0,
        "ArrDelayMinutes": 0.0, "Distance": 0.0,
        "Diverted": 0.0,
    }) \
    .withColumn("Year",      F.col("Year").cast(IntegerType())) \
    .withColumn("Month",     F.col("Month").cast(IntegerType())) \
    .withColumn("Distance",  F.col("Distance").cast(DoubleType())) \
    .withColumn("Cancelled", F.col("Cancelled").cast(IntegerType())) \
    .withColumn("DepDelay",  F.col("DepDelay").cast(DoubleType())) \
    .withColumn("ArrDelay",  F.col("ArrDelay").cast(DoubleType())) \
    .filter(F.col("Year").between(2022, 2025)) \
    .filter(F.col("Distance") > 0) \
    .withColumn("IsDelayed",
        (F.col("ArrDelay") > 15).cast(IntegerType())) \
    .withColumn("IsCancelled",
        (F.col("Cancelled") == 1).cast(IntegerType())) \
    .withColumn("TotalDelay",
        F.col("CarrierDelay") + F.col("WeatherDelay") +
        F.col("NASDelay") + F.col("SecurityDelay") +
        F.col("LateAircraftDelay"))

print("✅ Dropped nulls in core columns")
print("✅ Filled delay nulls with 0")
print("✅ Cast columns to correct types")
print("✅ Filtered years 2022-2025")
print("✅ Removed distance = 0 rows")
print("✅ Added IsDelayed column")
print("✅ Added IsCancelled column")
print("✅ Added TotalDelay column")
print(f"✅ Final columns : {len(df_clean.columns)}")
print("=" * 55)

print("\n📊 Sample data (5 rows):")
df_clean.select(
    "Year", "Month", "Reporting_Airline",
    "Origin", "Dest", "ArrDelay", "IsDelayed"
).show(5)

STEP 6 — DATA CLEANING
✅ Dropped nulls in core columns
✅ Filled delay nulls with 0
✅ Cast columns to correct types
✅ Filtered years 2022-2025
✅ Removed distance = 0 rows
✅ Added IsDelayed column
✅ Added IsCancelled column
✅ Added TotalDelay column
✅ Final columns : 34

📊 Sample data (5 rows):
+----+-----+-----------------+------+----+--------+---------+
|Year|Month|Reporting_Airline|Origin|Dest|ArrDelay|IsDelayed|
+----+-----+-----------------+------+----+--------+---------+
|2022|    1|               YX|   CMH| DCA|     4.0|        0|
|2022|    1|               YX|   CMH| DCA|   -24.0|        0|
|2022|    1|               YX|   CMH| DCA|   -13.0|        0|
|2022|    1|               YX|   CMH| DCA|     9.0|        0|
|2022|    1|               YX|   CMH| DCA|   -29.0|        0|
+----+-----+-----------------+------+----+--------+---------+
only showing top 5 rows


In [7]:
print("=" * 55)
print("STEP 7 — SAVING AS PARQUET")
print("=" * 55)
print("⏳ Writing to disk — please wait 5-10 minutes...\n")

start = time.time()

df_clean.write \
    .mode("overwrite") \
    .partitionBy("Year", "Month") \
    .parquet(PARQUET_PATH)

duration = round((time.time() - start) / 60, 2)

print(f"✅ Saved to     : {PARQUET_PATH}")
print(f"✅ Partitioned  : by Year and Month")
print(f"⏱️  Time taken   : {duration} minutes")
print("=" * 55)

STEP 7 — SAVING AS PARQUET
⏳ Writing to disk — please wait 5-10 minutes...

✅ Saved to     : D:\airline_pipeline\data\processed\flights_clean.parquet
✅ Partitioned  : by Year and Month
⏱️  Time taken   : 2.45 minutes


In [8]:
print("=" * 55)
print("STEP 8 — VERIFICATION")
print("=" * 55)

# ── count parquet files ───────────────────────────────────────────────
all_parquet = []
for root, dirs, files in os.walk(PARQUET_PATH):
    for f in files:
        if f.endswith(".parquet"):
            all_parquet.append(os.path.join(root, f))

total_gb = sum(os.path.getsize(f) for f in all_parquet) / (1024**3)

# ── count partitions ──────────────────────────────────────────────────
year_folders = [
    d for d in os.listdir(PARQUET_PATH)
    if d.startswith("Year=")
]

# ── read back and verify ──────────────────────────────────────────────
df_verify = spark.read.parquet(PARQUET_PATH)

print(f"✅ Parquet files saved   : {len(all_parquet)}")
print(f"✅ Total size on disk    : {total_gb:.2f} GB")
print(f"✅ Year partitions       : {sorted(year_folders)}")
print(f"✅ Columns in Parquet    : {len(df_verify.columns)}")
print(f"✅ Compression ratio     : {total_size_gb:.2f} GB → {total_gb:.2f} GB")
print(f"✅ Space saved           : {((1 - total_gb/total_size_gb)*100):.0f}%")
print("=" * 55)

print("\n📊 Sample from Parquet (reading back from disk):")
df_verify.select(
    "Year", "Month", "Reporting_Airline",
    "Origin", "Dest", "ArrDelay"
).show(5)

print("=" * 55)
print("🎉 STEP 5 COMPLETE!")
print("=" * 55)
print(f"  Source  : {len(csv_files)} CSV files ({total_size_gb:.2f} GB)")
print(f"  Output  : {len(all_parquet)} Parquet files ({total_gb:.2f} GB)")
print(f"  Years   : 2022, 2023, 2024, 2025")
print(f"  Columns : {len(df_verify.columns)}")
print("=" * 55)

STEP 8 — VERIFICATION
✅ Parquet files saved   : 109
✅ Total size on disk    : 0.39 GB
✅ Year partitions       : ['Year=2022', 'Year=2023', 'Year=2024', 'Year=2025']
✅ Columns in Parquet    : 34
✅ Compression ratio     : 11.40 GB → 0.39 GB
✅ Space saved           : 97%

📊 Sample from Parquet (reading back from disk):
+----+-----+-----------------+------+----+--------+
|Year|Month|Reporting_Airline|Origin|Dest|ArrDelay|
+----+-----+-----------------+------+----+--------+
|2024|    7|               MQ|   DFW| MLU|   -19.0|
|2024|    7|               MQ|   DFW| MLU|    -5.0|
|2024|    7|               MQ|   DFW| MLU|    71.0|
|2024|    7|               MQ|   DFW| MLU|    71.0|
|2024|    7|               MQ|   DFW| MLU|   -22.0|
+----+-----+-----------------+------+----+--------+
only showing top 5 rows
🎉 STEP 5 COMPLETE!
  Source  : 47 CSV files (11.40 GB)
  Output  : 109 Parquet files (0.39 GB)
  Years   : 2022, 2023, 2024, 2025
  Columns : 34


In [9]:
spark.stop()

print("=" * 55)
print("STEP 9 — SPARK STOPPED")
print("=" * 55)
print("✅ Spark session closed cleanly")
print("✅ All data saved to Parquet")
print("✅ Ready for Step 6 — Spark Processing!")
print("=" * 55)

STEP 9 — SPARK STOPPED
✅ Spark session closed cleanly
✅ All data saved to Parquet
✅ Ready for Step 6 — Spark Processing!
